## 1. Check GPU

In [1]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU! Go to Runtime > Change runtime type > T4 GPU")

CUDA available: True
GPU: Tesla T4


## 2. Install All Dependencies

In [2]:
# Install all dependencies
import subprocess
import sys

print("📦 Installing system dependencies...")
!apt-get update -qq
!apt-get install -y git wget net-tools > /dev/null 2>&1

print("📦 Installing Python packages...")

# Use CUDA 12.1 PyTorch (compatible with flash-attn)
!pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

# Install other dependencies (allow some flexibility)
!pip install -q "transformers>=4.45.0" "timm>=1.0.0" "accelerate>=1.0.0" "diffusers>=0.31.0"
!pip install -q websockets pyyaml opencv-python pillow "numpy>=1.26.4,<2.0" fvcore
!pip install -q einops thop cloudpickle easydict hydra-core
!pip install -q huggingface_hub
!pip install -q pyngrok ninja packaging

print("✅ All dependencies installed!")

📦 Installing system dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
📦 Installing Python packages...
📦 Installing Python packages...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.9 MB/s eta 0:00:0000:0100:01     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/799.0 MB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.9 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 134.6 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 134.6 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 113.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━

## 3. Clone Evo-1 and Download Checkpoint

In [3]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

# Clone Evo-1
if not Path("/content/Evo-1").exists():
    !git clone https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1
    print("✅ Evo-1 cloned")
    
    # Install Evo-1 requirements (but skip torch - we handle that separately)
    print("📦 Installing Evo-1 requirements...")
    !pip install -q -r /content/Evo-1/Evo_1/requirements.txt
    print("✅ Evo-1 requirements installed")
else:
    print("✅ Evo-1 already exists")

# Download checkpoint
checkpoint_dir = "/content/Evo-1/checkpoints/libero"
os.makedirs(checkpoint_dir, exist_ok=True)

if not Path(f"{checkpoint_dir}/config.json").exists():
    print("📥 Downloading checkpoint (1.4GB)...")
    snapshot_download(
        repo_id="MINT-SJTU/Evo1_LIBERO",
        local_dir=checkpoint_dir,
        local_dir_use_symlinks=False
    )
    print("✅ Checkpoint downloaded")
else:
    print("✅ Checkpoint already exists")

Cloning into '/content/Evo-1'...
remote: Enumerating objects: 705, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (23/23), done.
remote: Enumerating objects: 705, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 705 (delta 35), reused 17 (delta 17), pack-reused 665 (from 1)
Receiving objects: 100% (705/705), 6.46 MiB | 12.14 MiB/s, done.
Resolving deltas: 100% (186/186), done.
remote: Total 705 (delta 35), reused 17 (delta 17), pack-reused 665 (from 1)
Receiving objects: 100% (705/705), 6.46 MiB | 12.14 MiB/s, done.
Resolving deltas: 100% (186/186), done.
✅ Evo-1 cloned
📦 Installing Evo-1 requirements...
✅ Evo-1 cloned
📦 Installing Evo-1 requirements...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.5 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.5 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━

Cloning into '/content/Evo-1'...
remote: Enumerating objects: 705, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (23/23), done.
remote: Enumerating objects: 705, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 705 (delta 35), reused 17 (delta 17), pack-reused 665 (from 1)
Receiving objects: 100% (705/705), 6.46 MiB | 12.14 MiB/s, done.
Resolving deltas: 100% (186/186), done.
remote: Total 705 (delta 35), reused 17 (delta 17), pack-reused 665 (from 1)
Receiving objects: 100% (705/705), 6.46 MiB | 12.14 MiB/s, done.
Resolving deltas: 100% (186/186), done.
✅ Evo-1 cloned
📦 Installing Evo-1 requirements...
✅ Evo-1 cloned
📦 Installing Evo-1 requirements...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.5 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.5 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Cloning into '/content/Evo-1'...
remote: Enumerating objects: 705, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (23/23), done.
remote: Enumerating objects: 705, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 705 (delta 35), reused 17 (delta 17), pack-reused 665 (from 1)
Receiving objects: 100% (705/705), 6.46 MiB | 12.14 MiB/s, done.
Resolving deltas: 100% (186/186), done.
remote: Total 705 (delta 35), reused 17 (delta 17), pack-reused 665 (from 1)
Receiving objects: 100% (705/705), 6.46 MiB | 12.14 MiB/s, done.
Resolving deltas: 100% (186/186), done.
✅ Evo-1 cloned
📦 Installing Evo-1 requirements...
✅ Evo-1 cloned
📦 Installing Evo-1 requirements...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.5 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.5 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━

...

config.json:   0%|          | 0.00/990 [00:00<?, ?B/s]

checkpoint.json:   0%|          | 0.00/89.0 [00:00<?, ?B/s]

norm_stats.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

mp_rank_00_model_states.pt:   0%|          | 0.00/1.55G [00:00<?, ?B/s]

✅ Checkpoint downloaded


In [4]:
# ⚠️ CRITICAL: Install flash-attn for proper model performance
# This cell installs PyTorch 2.4.0+cu121 which is compatible with flash-attn

print("🔧 Installing flash-attn with compatible PyTorch...")

# Step 1: Install PyTorch 2.4.0 + CUDA 12.1 (compatible with flash-attn)
print("\n📦 Step 1: Installing PyTorch 2.4.0+cu121...")
!pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

# Step 2: Install flash-attn
print("\n📦 Step 2: Installing flash-attn...")
!pip install flash-attn --no-build-isolation 2>&1 | tail -5

print("\n" + "="*60)
print("⚠️  IMPORTANT: RESTART RUNTIME NOW!")
print("="*60)
print("\nGo to: Runtime > Restart runtime")
print("Then run cells starting from cell 9 (Verify Dependencies)")
print("="*60)

🔧 Installing flash-attn with compatible PyTorch...

📦 Step 1: Installing PyTorch 2.4.0+cu121...

📦 Step 2: Installing flash-attn...

📦 Step 2: Installing flash-attn...
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=255985226 sha256=ee1fbb7dc9d4f6e973687e15e245727d39c2b5f5884c74fa30d21bd1841854af
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=255985226 sha256=ee1fbb7dc9d4f6e973687e15e245727d39c2b5f5884c74fa30d21bd1841854af
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn

⚠️  IMPORTANT: RESTART RUNTIME NOW!

Go to: Runtime > Restart runtime
Then run cells starting from cell 9 (Verify Dependencies)

⚠️  IMPORTANT: RESTART RUNTIME NOW!

Go to: Runtime > Restart runtime
Then run c

In [ ]:
# Skip this cell - requirements already installed in previous cells
# DO NOT reinstall requirements.txt as it may override PyTorch version
print("⏭️  Skipping - dependencies already installed")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.4.0+cu121 requires torch==2.4.0, but you have torch 2.5.1 which is incompatible.


In [1]:
# ✅ Run this cell AFTER restarting the runtime
# Verify torch and flash-attn are working

import torch
print(f"✅ PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

try:
    import flash_attn
    print(f"✅ flash-attn: {flash_attn.__version__}")
except ImportError:
    print("❌ flash-attn not available")
    print("   Re-run cell 7 (flash-attn installation) and restart runtime again")

# Verify CUDA works
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ CUDA not available!")

print("\n✅ Ready to continue with remaining cells")

✅ PyTorch: 2.4.0+cu121, CUDA: 12.1
✅ flash-attn: 2.8.3
✅ GPU: Tesla T4

✅ Ready to continue with remaining cells


## 4. Setup Evo-1 Server

In [2]:
import json

# Update config to use CUDA
config_path = "/content/Evo-1/checkpoints/libero/config.json"
with open(config_path, 'r') as f:
    config = json.load(f)
config['device'] = 'cuda'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

# Update server checkpoint path AND port
server_file = "/content/Evo-1/Evo_1/scripts/Evo1_server.py"
with open(server_file, 'r') as f:
    content = f.read()

# Fix checkpoint path
content = content.replace(
    'ckpt_dir = "Your/Path/To/Checkpoint"',
    'ckpt_dir = "/content/Evo-1/checkpoints/libero"'
)

# Change port from 9000 to 9001 (Colab uses 9000 for Jupyter)
content = content.replace('port = 9000', 'port = 9001')
content = content.replace('port=9000', 'port=9001')
content = content.replace(':9000', ':9001')

with open(server_file, 'w') as f:
    f.write(content)

print("✅ Evo-1 server configured (using port 9001)")

# Patch InternVL3 to disable flash_attn (workaround for CUDA compatibility)
# The model's flash_attn implementation has issues with certain CUDA versions
print("🔧 Patching InternVL3 to disable flash_attn (CUDA compatibility fix)...")

internvl_cache = "/root/.cache/huggingface/modules/transformers_modules/OpenGVLab"
import os
import glob

# Find the InternVL3 model files
for model_dir in glob.glob(f"{internvl_cache}/InternVL3*"):
    vit_file = os.path.join(model_dir, "modeling_intern_vit.py")
    if os.path.exists(vit_file):
        with open(vit_file, 'r') as f:
            vit_content = f.read()
        
        # Force use_flash_attn to False in InternAttention
        if "self.use_flash_attn = use_flash_attn" in vit_content:
            vit_content = vit_content.replace(
                "self.use_flash_attn = use_flash_attn",
                "self.use_flash_attn = False  # Disabled for CUDA compatibility"
            )
            with open(vit_file, 'w') as f:
                f.write(vit_content)
            print(f"  ✅ Patched: {vit_file}")

print("✅ InternVL3 patched to use naive attention (more compatible)")

✅ Evo-1 server configured (using port 9001)


## 5. Install Miniconda and Setup LIBERO Environment

In [3]:
%%bash
set -e

# Install Miniconda if not already installed
if [ ! -f /usr/local/bin/conda ]; then
    echo "📦 Installing Miniconda..."
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
    bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local -f
    rm Miniconda3-latest-Linux-x86_64.sh
    /usr/local/bin/conda init bash
    echo "✅ Miniconda installed"
else
    echo "✅ Conda already installed"
fi

# Accept conda Terms of Service
echo "📝 Accepting conda Terms of Service..."
/usr/local/bin/conda config --set allow_conda_downgrades true
yes | /usr/local/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
yes | /usr/local/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true

# Create LIBERO environment
if [ ! -d /usr/local/envs/libero ]; then
    echo "📦 Creating LIBERO environment (Python 3.8.13)..."
    /usr/local/bin/conda create -p /usr/local/envs/libero python=3.8.13 -y
    echo "✅ Environment created"
else
    echo "✅ Environment already exists"
fi

📦 Installing Miniconda...
PREFIX=/usr/local
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /usr/local
no change     /usr/local/condabin/conda
no change     /usr/local/bin/conda
no change     /usr/local/bin/conda-env
no change     /usr/local/bin/activate
no change     /usr/local/bin/deactivate
no change     /usr/local/etc/profile.d/conda.sh
no change     /usr/local/etc/fish/conf.d/conda.fish
no change     /usr/local/shell/condabin/Conda.psm1
no change     /usr/local/shell/condabin/conda-hook.ps1
no change     /usr/local/lib/python3.13/s



==> WARNING: A newer version of conda exists. <==
    current version: 25.9.1
    latest version: 25.11.0

Please update conda by running

    $ conda update -n base -c defaults conda




## 6. Clone and Install LIBERO

In [4]:
%%bash
set -e

# Clone LIBERO
mkdir -p /content/Evo-1/LIBERO_evaluation
cd /content/Evo-1/LIBERO_evaluation

if [ ! -d "LIBERO" ]; then
    echo "📦 Cloning LIBERO..."
    git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git
    echo "✅ LIBERO cloned"
else
    echo "✅ LIBERO already exists"
fi

cd LIBERO

# Install dependencies in conda environment (use full path to conda)
echo "📦 Installing LIBERO dependencies..."
/usr/local/bin/conda run -p /usr/local/envs/libero pip install -q -r requirements.txt
/usr/local/bin/conda run -p /usr/local/envs/libero pip install -q torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 --extra-index-url https://download.pytorch.org/whl/cu113
/usr/local/bin/conda run -p /usr/local/envs/libero pip install -q -e .
/usr/local/bin/conda run -p /usr/local/envs/libero pip install -q websockets huggingface_hub

echo "✅ LIBERO installed"

📦 Cloning LIBERO...
✅ LIBERO cloned
📦 Installing LIBERO dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.2/829.2 kB 34.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.5/734.5 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 121.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 6

Cloning into 'LIBERO'...
Updating files: 100% (1116/1116), done.
  DEPRECATION: Legacy editable install of libero==0.1.0 from file:///content/Evo-1/LIBERO_evaluation/LIBERO (setup.py develop) is deprecated. pip 25.0 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457



## 7. Upload Your Client Script

Upload `libero_client_4tasks.py` to `/content/Evo-1/LIBERO_evaluation/LIBERO/`

Or use the file upload button in Colab.

## 8. Setup ngrok (Free - No Account)

Get your free ngrok authtoken from: https://dashboard.ngrok.com/get-started/your-authtoken

Paste it below:

In [3]:
# Set your ngrok authtoken here
NGROK_AUTH_TOKEN = "35srHlTDCClg3rKtjzoIzHBfxTZ_7WFCjvg5VHTd8As6k7kfH"  # Get free token from https://dashboard.ngrok.com/

from pyngrok import ngrok, conf
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("✅ ngrok configured")

✅ ngrok configured                                                                                  
✅ ngrok configured


## 9. Kill Any Existing Server (Optional)
Run this only if you need to restart the server.

In [9]:
# Kill any existing Evo-1 server AND ngrok tunnels
# Run this BEFORE starting the server if you get "address already in use" or ngrok errors
# NOTE: We use port 9001 because Colab's Jupyter uses port 9000

import subprocess
import time

print("🔄 Killing existing ngrok tunnels...")
try:
    from pyngrok import ngrok
    ngrok.kill()
    print("✅ ngrok tunnels killed")
except:
    print("⚠️ ngrok not running or not installed")

print("🔄 Killing any existing Evo-1 server...")

# Kill by process name (safe - won't affect Jupyter)
!pkill -9 -f "Evo1_server" 2>/dev/null || true

# Kill anything on port 9001 (our server port, NOT 9000 which is Jupyter)
!kill -9 $(lsof -t -i:9001) 2>/dev/null || true

time.sleep(2)

# Verify port is free
result = !netstat -tuln 2>/dev/null | grep :9001
if result:
    print("⚠️ Port 9001 still in use!")
else:
    print("✅ Port 9001 is free, ready to start server")

✅ Port 9001 is free, ready to start server


In [10]:
# Force disconnect ALL ngrok tunnels - run this first!
from pyngrok import ngrok
import subprocess

# Method 1: Disconnect all active tunnels via API
try:
    tunnels = ngrok.get_tunnels()
    print(f"Found {len(tunnels)} active tunnel(s)")
    for tunnel in tunnels:
        print(f"  Disconnecting: {tunnel.public_url}")
        ngrok.disconnect(tunnel.public_url)
except Exception as e:
    print(f"API disconnect failed: {e}")

# Method 2: Kill local ngrok process
ngrok.kill()
print("✅ Local ngrok process killed")

# Method 3: Also kill any system ngrok processes
!pkill -9 ngrok 2>/dev/null || true
print("✅ System ngrok processes killed")

print("\n⏳ Wait 10-15 seconds for ngrok servers to release the endpoint...")
print("   Then run the server startup cell")

Found 0 active tunnel(s)
✅ Local ngrok process killed
✅ System ngrok processes killed

⏳ Wait 10-15 seconds for ngrok servers to release the endpoint...
   Then run the server startup cell
✅ System ngrok processes killed

⏳ Wait 10-15 seconds for ngrok servers to release the endpoint...
   Then run the server startup cell


## 10. Run Evo-1 Server

In [11]:
import subprocess
import time
from pyngrok import ngrok
import os

SERVER_PORT = 9001  # Using 9001 because Colab's Jupyter uses 9000

# Ensure all server dependencies are installed (but DO NOT reinstall torch!)
print("📦 Installing server dependencies (keeping existing PyTorch)...")
!pip install -q websockets opencv-python pillow transformers timm accelerate diffusers fvcore einops

# Verify flash-attn is available
import torch
print(f"📌 PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
try:
    import flash_attn
    print(f"✅ flash-attn: {flash_attn.__version__} - Model will use FlashAttention!")
except ImportError:
    print("⚠️  flash-attn not available - model will use slower attention")
    print("   Consider re-running cell 7 and restarting runtime")

# Start Evo-1 server in background
print("\n🚀 Starting Evo-1 server (takes 2-3 minutes to load model)...")
with open('/content/server.log', 'w') as log_file:
    server_process = subprocess.Popen(
        ["python", "Evo_1/scripts/Evo1_server.py"],
        cwd="/content/Evo-1",
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

print(f"✅ Server started (PID: {server_process.pid})")
print("📝 Logs: /content/server.log")

# Wait and check if server is actually running
for i in range(24):  # Check every 5 seconds for 2 minutes
    time.sleep(5)
    # Check if process is still alive
    if server_process.poll() is not None:
        print(f"\n❌ Server crashed! Exit code: {server_process.returncode}")
        print("\n📋 Last 50 lines of server log:")
        !tail -50 /content/server.log
        raise Exception("Server failed to start")
    
    # Check if port 9001 is listening
    result = !netstat -tuln 2>/dev/null | grep :9001 || true
    if result:
        print(f"\n✅ Server listening on port {SERVER_PORT} after {(i+1)*5} seconds")
        break
    
    if i % 3 == 0:
        print(f"⏳ Waiting... ({(i+1)*5}s elapsed)")

# Start ngrok tunnel
print("\n🌐 Starting ngrok tunnel...")
tunnel = ngrok.connect(SERVER_PORT, bind_tls=True)
public_url = tunnel.public_url.replace('https://', 'wss://')

print("\n" + "="*60)
print("✅ SERVER READY!")
print("="*60)
print(f"\n🌐 Public URL: {public_url}")
print("\n📝 For Mac client, update SERVER_URL to:")
print(f"   SERVER_URL = '{public_url}'")
print("\n" + "="*60)

# Check if client file exists (included in Evo-1 repo)
client_file = "/content/Evo-1/LIBERO_evaluation/libero_client_4tasks.py"
if os.path.exists(client_file):
    print("\n✅ Client file found")
    print("\nTo run client on Colab, execute next cell")
    print("Or run from your Mac with the URL above")
else:
    print("\n⚠️  Client file not found in Evo-1 repo!")

print("\n⚠️  Keep this cell running during evaluation!")


✅ Server listening on port 9001 after 25 seconds

🌐 Starting ngrok tunnel...

✅ SERVER READY!

🌐 Public URL: wss://semisolemnly-unforecasted-allie.ngrok-free.dev

📝 For Mac client, update SERVER_URL to:
   SERVER_URL = 'wss://semisolemnly-unforecasted-allie.ngrok-free.dev'


✅ Client file found

To run client on Colab, execute next cell
Or run from your Mac with the URL above

⚠️  Keep this cell running during evaluation!

✅ SERVER READY!

🌐 Public URL: wss://semisolemnly-unforecasted-allie.ngrok-free.dev

📝 For Mac client, update SERVER_URL to:
   SERVER_URL = 'wss://semisolemnly-unforecasted-allie.ngrok-free.dev'


✅ Client file found

To run client on Colab, execute next cell
Or run from your Mac with the URL above

⚠️  Keep this cell running during evaluation!


In [ ]:
!ls /content

In [ ]:
!cat /content/Evo-1/Evo_1/scripts/Evo1_server.py
!ls /content/Evo-1/checkpoints/libero

In [13]:
!cat /content/server.log

`torch_dtype` is deprecated! Use `dtype` instead!
2025-11-28 12:28:27.059083: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764332907.077494   11437 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764332907.084632   11437 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764332907.102454   11437 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764332907.102482   11437 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764332907.102484   11437

## 10. Run Client on Colab (Optional)

In [ ]:
# Update client to use localhost
client_file = "/content/Evo-1/LIBERO_evaluation/libero_client_4tasks.py"

if os.path.exists(client_file):
    with open(client_file, 'r') as f:
        client_content = f.read()
    
    # Update SERVER_URL to localhost
    import re
    client_content = re.sub(
        r'SERVER_URL = "[^"]*"',
        'SERVER_URL = "ws://localhost:9000"',
        client_content
    )
    
    with open(client_file, 'w') as f:
        f.write(client_content)
    
    print("✅ Client configured for localhost")
    print("🚀 Running LIBERO evaluation (2-3 hours)...\n")
    
    # Run client
    !cd /content/Evo-1/LIBERO_evaluation/LIBERO && conda run -p /usr/local/envs/libero python libero_client_4tasks.py
else:
    print("⚠️ Client file not found!")

## 11. Check Server Logs

In [ ]:
!tail -30 /content/server.log

In [ ]:
for i in range(100):
    print(i, end = '\t')

## 12. Stop Server (When Done)

In [ ]:
# Stop server and ngrok
try:
    server_process.terminate()
    server_process.wait(timeout=5)
    print("✅ Server stopped")
except:
    !fuser -k 9001/tcp 2>/dev/null || pkill -f Evo1_server.py
    print("✅ Server killed")

ngrok.kill()
print("✅ ngrok stopped")